In [ ]:
"""Instant Insanity: hybrid split with a QAOA sampler and a Grover sampler.

THE SPLIT
  Quantum   : a circuit over the front/back variables only (3 qubits per cube, one-hot,
              W-state start) produces candidate picks, one of 3 opposite pairs per cube.
              Two interchangeable samplers are provided:
                * QAOA   - constrained QAOA (W states + XY mixer), angles tuned classically
                * Grover - amplitude amplification with a sign-flip oracle on "cost == 0"
  Classical : check each sampled pick is a 2-factor, keep the distinct ones, pair two that
              share no edge on any cube, build the tower and verify it with solves().

WHAT IS SIMULATED
  Both samplers are simulated exactly inside the one-hot subspace (3^N states), so no
  quantum SDK is needed to run this file. The Grover oracle is an IDEAL sign flip computed
  classically; building it as a circuit (reversible degree counting) is not done here.
  QAOA's simulation matches the starter notebook's warm-up numbers (checked at N=4).

HONEST FRAMING
  The quantum stage solves an easy sub-problem: there are only 3^N front/back picks, and a
  laptop enumerates them instantly. Nothing here is evidence of quantum advantage.

USE
  python hybrid_grover.py            # N=4 starter puzzle: classical refs, QAOA, Grover
  python hybrid_grover.py --cubes 8  # full run on 8 cubes (double the starter)
  python hybrid_grover.py --scale    # also N=5..8 with the notebook's unique-answer seeds
In a notebook, import the pieces: assemble_from_counts(cubes, counts) takes the counts
returned by run_on_qbraid() for the 12-qubit front/back circuit.
"""
import argparse
import itertools
import random
import time

import numpy as np
from scipy.optimize import minimize

SEED = 7
DEFAULT_CUBES = 8      # what the Run button uses (no command-line flags there); set 4 for the starter
DEFAULT_SCALE = False  # True also runs the N=5..8 scaling table

CUBES = [["GR", "BR", "GY"], ["GY", "RB", "BY"], ["BR", "RG", "RY"], ["YR", "RG", "BG"]]
BONUS = {5: 16, 6: 86, 7: 68, 8: 4}      # seeds with exactly one tower (starter notebook)

DEVICE_ID = "aws:aws:sim:sv1"
# =============================================================================
# 1. Puzzle, classical solvers, and the notebook's tower helpers
# =============================================================================
def colours_of(cubes):
    return sorted({ch for cube in cubes for pair in cube for ch in pair})


def make_puzzle(n, seed):
    """Copied from the starter notebook: N cubes, N colours, a planted answer."""
    rng = random.Random(seed)
    letters = "".join(chr(ord("A") + i) for i in range(n))
    cycles = []
    for _ in range(2):
        order = rng.sample(letters, n)
        lines = [order[i] + order[(i + 1) % n] for i in range(n)]
        rng.shuffle(lines)
        cycles.append(lines)
    cubes = []
    for i in range(n):
        third = rng.choice(letters) + rng.choice(letters)
        pairs = ["".join(rng.sample(pair, 2)) for pair in (cycles[0][i], cycles[1][i], third)]
        rng.shuffle(pairs)
        cubes.append(pairs)
    return cubes


def orientations(cube):
    """The 24 ways a cube can sit, as (front, back, left, right) colours (notebook)."""
    views = []
    for a, b in itertools.permutations(range(3), 2):
        for fa in (0, 1):
            for fb in (0, 1):
                f, l = cube[a], cube[b]
                views.append((f[fa], f[1 - fa], l[fb], l[1 - fb]))
    return views


def solves(tower):
    """True when each of the four sides shows every colour once (notebook)."""
    return all(len({view[side] for view in tower}) == len(tower) for side in range(4))


try:                                          # in the repo, use the notebook's own helpers
    from resources.helpers import tower_from
except Exception:                             # fallback: same logic as resources/helpers.py
    def _orient(cubes, picks):
        lines = {c: cubes[c][p] for c, p in enumerate(picks)}
        placed = {}
        while len(placed) < len(lines):
            c = next(c for c in lines if c not in placed)
            here = lines[c][0]
            while c is not None:
                u, v = lines[c]
                placed[c] = (here, v if u == here else u)
                here = placed[c][1]
                c = next((d for d, (a, b) in lines.items()
                          if d not in placed and here in (a, b)), None)
        return placed

    def tower_from(cubes, choice):
        """One (front/back pair, left/right pair) choice per cube -> tower."""
        front = _orient(cubes, [fb for fb, _ in choice])
        side = _orient(cubes, [lr for _, lr in choice])
        return [(front[c][0], side[c][0], front[c][1], side[c][1]) for c in range(len(cubes))]


def is_two_factor(cubes, picks):
    deg = dict.fromkeys(colours_of(cubes), 0)
    for cube, p in zip(cubes, picks):
        for ch in cube[p]:
            deg[ch] += 1
    return all(d == 2 for d in deg.values())


def two_factors(cubes):
    return [p for p in itertools.product(range(3), repeat=len(cubes)) if is_two_factor(cubes, p)]


def disjoint(f, g):
    """No cube uses the same opposite pair for both axes."""
    return all(a != b for a, b in zip(f, g))


def graph_solve(cubes):
    """Classical graph method: all 2-factors, then a disjoint pair. Returns a tower or None."""
    factors = two_factors(cubes)
    for f, g in itertools.permutations(factors, 2):
        if disjoint(f, g):
            tower = tower_from(cubes, list(zip(f, g)))
            if solves(tower):
                return tower
    return None


def brute_force(cubes):
    """All 24^N stackings. Only sensible for N <= 4."""
    options = [orientations(c) for c in cubes]
    return [t for t in itertools.product(*options) if solves(t)]


# =============================================================================
# 2. Front/back cost on the valid subspace (shared by both samplers)
# =============================================================================
def fb_costs(cubes):
    """Degree-rule cost on the 3^N one-hot picks, flattened (index = base-3 digits, cube 0 first).
    Equals the notebook's front_back_hamiltonian (lam=0) on valid states; zero = a 2-factor."""
    n, cols = len(cubes), colours_of(cubes)
    costs = np.zeros((3,) * n)
    for picks in itertools.product(range(3), repeat=n):
        costs[picks] = sum((sum(cubes[i][p].count(k) for i, p in enumerate(picks)) - 2) ** 2
                           for k in cols)
    return costs.reshape(-1)


# =============================================================================
# 3. Quantum sampler A: constrained QAOA (simulated in the subspace)
# =============================================================================
def block_mixer(beta):
    """XXPlusYYGate(2*beta) on qubit pairs (0,1),(0,2),(1,2) of a one-hot 3-qubit block."""
    c, s = np.cos(beta), -1j * np.sin(beta)
    M = np.eye(3, dtype=complex)
    for i, j in ((0, 1), (0, 2), (1, 2)):
        U = np.eye(3, dtype=complex)
        U[i, i] = U[j, j] = c
        U[i, j] = U[j, i] = s
        M = U @ M
    return M


def qaoa_runner(costs, n):
    size = 3 ** n

    def run(gammas, betas):
        psi = np.full(size, size ** -0.5, dtype=complex)            # W state on every block
        for g, b in zip(gammas, betas):
            psi = (psi * np.exp(-1j * g * costs)).reshape((3,) * n)  # cost layer
            M = block_mixer(b)                                        # mixer layer
            for axis in range(n):
                psi = np.moveaxis(np.tensordot(M, psi, axes=([1], [axis])), 0, axis)
            psi = psi.reshape(-1)
        return np.abs(psi) ** 2
    return run


def tune_qaoa(costs, n, pmax, rng, restarts=4):
    """Classical tuning: p=1 grid scan (as in the notebook), then grow one layer at a time
    with COBYLA, minimising the average cost. Returns {p: (gammas, betas)}."""
    run = qaoa_runner(costs, n)
    best = min((run([g], [b]) @ costs, g, b)
               for g in np.linspace(0, np.pi, 41) for b in np.linspace(0, np.pi, 41))
    g, b = np.array([best[1]]), np.array([best[2]])
    angles = {1: (g.copy(), b.copy())}
    for p in range(2, pmax + 1):
        winner = None
        for r in range(restarts):
            new = (0.0, 0.0) if r == 0 else (rng.uniform(0, np.pi), rng.uniform(0, np.pi / 2))
            res = minimize(lambda x: run(x[:p], x[p:]) @ costs, np.r_[g, new[0], b, new[1]],
                           method="COBYLA", options={"maxiter": 250})
            if winner is None or res.fun < winner.fun:
                winner = res
        g, b = winner.x[:p], winner.x[p:]
        angles[p] = (g.copy(), b.copy())
    return angles


# =============================================================================
# 4. Quantum sampler B: Grover / amplitude amplification (ideal oracle)
# =============================================================================
def grover_iterations(size, marked):
    """Rounds that maximise success when the number of solutions is known."""
    theta = np.arcsin(np.sqrt(marked / size))
    return max(1, int(round(np.pi / (4 * theta) - 0.5)))


def grover_probs(costs, rounds):
    """Start from the even blend over valid picks (the W-state start). Each round: the oracle
    flips the sign of every cost-0 pick, then diffusion reflects about the average (which is
    the reflection about the W-state start inside the valid subspace)."""
    good = costs == 0
    psi = np.full(len(costs), len(costs) ** -0.5)
    for _ in range(rounds):
        psi = np.where(good, -psi, psi)
        psi = 2 * psi.mean() - psi
    return psi ** 2


def bbht_oracle_calls(size, marked, rng):
    """Unknown number of solutions (Boyer-Brassard-Hoyer-Tapp): try a random number of rounds
    below a cap that grows by 6/5 each failure. Returns the oracle calls used until a measured
    pick passes the classical check. `marked` is used only to simulate the measurement, not to
    choose the round counts."""
    theta = np.arcsin(np.sqrt(marked / size))
    cap, calls = 1.0, 0
    while True:
        j = int(rng.integers(0, int(np.ceil(cap))))
        calls += j + 1                                 # j rounds plus one classical check
        if rng.random() < np.sin((2 * j + 1) * theta) ** 2:
            return calls
        cap = min(1.2 * cap, np.sqrt(size))


# =============================================================================
# 5. Classical assembly
# =============================================================================
def assemble_from_indices(cubes, costs, shot_indices):
    """Walk shots in order; return (shots used, tower) at the first verified tower."""
    n = len(cubes)
    found = []
    for used, idx in enumerate(shot_indices, 1):
        if costs[idx] != 0:
            continue
        picks = tuple(int(d) for d in np.unravel_index(idx, (3,) * n))
        if picks in found:
            continue
        for other in found:
            if disjoint(picks, other):
                tower = tower_from(cubes, [(picks[c], other[c]) for c in range(n)])
                if solves(tower):
                    return used, tower
        found.append(picks)
    return None, None


def decode_key(key, n):
    """Qiskit counts key for the 3-qubits-per-cube layout (qubit 0 is the RIGHTMOST character).
    Returns one pair index per cube, or None if some cube is not one-hot."""
    bits = key[::-1]
    blocks = [bits[3 * c:3 * c + 3] for c in range(n)]
    if any(b.count("1") != 1 for b in blocks):
        return None
    return tuple(b.index("1") for b in blocks)


def encode_key(picks):
    """Inverse of decode_key; used to test the decoder on a known answer."""
    bits = ["0"] * (3 * len(picks))
    for c, p in enumerate(picks):
        bits[3 * c + p] = "1"
    return "".join(reversed(bits))


def assemble_from_counts(cubes, counts):
    """Classical stage for real circuit output. counts: {Qiskit bitstring: shots}.
    Returns (towers verified by solves(), number of distinct valid 2-factors seen)."""
    n = len(cubes)
    found = []
    for key in counts:
        picks = decode_key(key, n)
        if picks is not None and is_two_factor(cubes, picks) and picks not in found:
            found.append(picks)
    towers = []
    for f, g in itertools.permutations(found, 2):
        if disjoint(f, g):
            tower = tower_from(cubes, list(zip(f, g)))
            if solves(tower):
                towers.append(tower)
    return towers, len(found)


# =============================================================================
# 6. Experiments
# =============================================================================
def shots_to_tower(cubes, costs, probs, trials, rng, max_shots=500):
    """Shots until a verified tower, over independent simulated runs (inf if none)."""
    out = []
    for _ in range(trials):
        draws = rng.choice(len(probs), size=max_shots, p=probs / probs.sum())
        used, _ = assemble_from_indices(cubes, costs, draws)
        out.append(used if used is not None else np.inf)
    return np.array(out, dtype=float)


def shots_for_confidence(pa, pb, conf=0.95):
    """Shots to see both of two specific picks at least once with probability conf."""
    def seen(s):
        return 1 - (1 - pa) ** s - (1 - pb) ** s + (1 - pa - pb) ** s
    if pa <= 0 or pb <= 0 or seen(1e12) < conf:
        return float("inf")
    lo = hi = 1
    while seen(hi) < conf:
        hi *= 2
    while lo < hi:
        mid = (lo + hi) // 2
        lo, hi = (lo, mid) if seen(mid) >= conf else (mid + 1, hi)
    return int(lo)


def needed_pair(cubes):
    """The two front/back 2-factors a solution needs (None unless exactly one disjoint pair)."""
    factors = two_factors(cubes)
    pairs = [(f, g) for f, g in itertools.combinations(factors, 2) if disjoint(f, g)]
    return pairs[0] if len(pairs) == 1 else None


def summarise(label, cubes, costs, probs, trials, rng):
    """Print one sampler's results and return them as a dict for the slide summary."""
    n = len(cubes)
    good = costs == 0
    runs = shots_to_tower(cubes, costs, probs, trials, rng)
    ok = np.isfinite(runs)
    print(f"  {label}: P(valid 2-factor per shot) {probs[good].sum():.1%} "
          f"(random valid pick {good.mean():.1%})")
    if ok.any():
        print(f"      shots to a verified tower over {trials} runs: median {np.median(runs):.0f}, "
              f"mean {runs[ok].mean():.0f}, 95th pct {np.sort(runs)[int(0.95 * len(runs)) - 1]:.0f}"
              + ("" if ok.all() else f"  ({(~ok).sum()} runs found none in 500 shots)"))
    else:
        print("      no run found a tower within 500 shots")
    return {"label": label, "p_valid": probs[good].sum(), "p_random": good.mean(),
            "solved": ok.mean(), "median": np.median(runs) if ok.any() else np.inf,
            "mean": runs[ok].mean() if ok.any() else np.inf,
            "p95": np.sort(runs)[int(0.95 * len(runs)) - 1],
            "within": {s: (runs <= s).mean() for s in (10, 50, 100)}}


def print_slide_stats(st):
    """One block with every number worth putting on a slide."""
    n, line = st["n"], "=" * 78
    pct = lambda x: f"{100 * x:.1f}%"
    shots = lambda x: "none" if not np.isfinite(x) else f"{x:.0f}"
    print(f"\n{line}\nSLIDE STATS  ·  N={n} cubes, {n} colours  ·  {st['puzzle']}\n{line}")

    print("\nThe search space")
    print(f"  all stackings (24^N)                 {24 ** n:>16,}")
    print(f"  after the graph trick (6^N)          {6 ** n:>16,}")
    print(f"  front/back picks the circuit sees    {3 ** n:>16,}   (3^N)")
    print(f"  front/back picks that are 2-factors  {st['factors']:>16,}   "
          f"({pct(st['factors'] / 3 ** n)} of picks)")
    print(f"  distinct solved towers               {st['pairs']:>16,}   (shows as {2 * st['pairs']} "
          f"(f, g) cases: the tower and its quarter turn)")
    print(f"  random stacking solves it            {st['towers'] / 24 ** n:>16.2e}   "
          f"(1 in {24 ** n / st['towers']:,.0f})" if st["towers"] else "")

    print("\nQuantum resources (front/back circuit)")
    print(f"  qubits                               {3 * n:>16}   (3 per cube, one-hot)")
    print(f"  Grover iterations (known count)      {st['grover_k']:>16}")
    print(f"  BBHT oracle calls (unknown count)    {st['bbht']:>16.1f}   (mean, to one 2-factor)")
    print("  note: ideal oracle, so oracle ancilla qubits and gates are not counted here")

    print("\nSuccess rates")
    print(f"  {'method':<20}{'P(2-factor)':>13}{'vs':>8}{'solved in':>11}{'median':>8}"
          f"{'95th pct':>10}{'solved in':>11}{'solved in':>11}")
    print(f"  {'':<20}{'per shot':>13}{'random':>8}{'500 shots':>11}{'shots':>8}"
          f"{'shots':>10}{'10 shots':>11}{'50 shots':>11}")
    rows = [{"label": "random valid pick", "p_valid": st["p_random"], "p_random": st["p_random"],
             **st["random_runs"]}] + st["samplers"]
    for r in rows:
        print(f"  {r['label']:<20}{pct(r['p_valid']):>13}{r['p_valid'] / r['p_random']:>7.1f}x"
              f"{pct(r['solved']):>11}{shots(r['median']):>8}{shots(r['p95']):>10}"
              f"{pct(r['within'][10]):>11}{pct(r['within'][50]):>11}")

    print("\nClassical baselines")
    if st["bf_time"] is not None:
        print(f"  brute force over 24^N stackings      {st['bf_time']:>15.2f}s")
    else:
        print(f"  brute force over 24^N stackings      {'skipped':>16}   (too many)")
    print(f"  graph method (enumerate 3^N picks)   {1000 * st['graph_time']:>14.1f}ms")

    print(f"\n1000 Grover shots -> {st['seen']} distinct 2-factors seen, "
          f"{st['verified']} verified tower(s)")
    print("Honest framing: simulated in the valid subspace; at this size the classical graph\n"
          "method is instant, so this shows the method working, not quantum advantage.")
    print(line)


def run_n4(trials, rng, n=4):
    """Main experiment for any N. N=4 is the starter puzzle; larger N uses make_puzzle with
    the notebook's unique-answer seed."""
    cubes = CUBES if n == 4 else make_puzzle(n, BONUS[n])
    print("=" * 78)
    print(f"N={n} " + ("starter puzzle" if n == 4 else f"generated puzzle (seed {BONUS[n]})"))
    print("=" * 78)
    st = {"n": n, "puzzle": "starter puzzle" if n == 4 else f"make_puzzle seed {BONUS[n]}",
          "bf_time": None, "samplers": []}
    if n <= 4:
        t = time.perf_counter()
        bf = brute_force(cubes)
        st["bf_time"] = time.perf_counter() - t
        print(f"classical brute force : {24 ** n:,} stackings, {len(bf)} solved, "
              f"{st['bf_time']:.2f} s")
    else:
        bf = None
        print(f"classical brute force : {24 ** n:,} stackings, skipped (too many to enumerate)")
    t = time.perf_counter()
    tower = graph_solve(cubes)
    st["graph_time"] = time.perf_counter() - t
    factors = two_factors(cubes)
    st["factors"] = len(factors)
    st["pairs"] = sum(disjoint(f, g) for f, g in itertools.combinations(factors, 2))
    st["towers"] = len(bf) if bf is not None else 8 * st["pairs"]   # 8 symmetric views each
    print(f"classical graph method: {3 ** n} front/back picks -> {len(factors)} 2-factors -> "
          f"tower {'found' if tower else 'NOT found'}, {1000 * (time.perf_counter() - t):.1f} ms, "
          f"verified {bool(tower) and solves(tower)}")

    costs = fb_costs(cubes)

    print(f"\nQuantum stage A: constrained QAOA, {3 * n} qubits (simulated)")
    angles = tune_qaoa(costs, n, pmax=3, rng=rng)
    for p in (2, 3):
        probs = qaoa_runner(costs, n)(*angles[p])
        st["samplers"].append(summarise(f"QAOA p={p}", cubes, costs, probs, trials, rng))

    print(f"\nQuantum stage B: Grover, {3 * n} qubits, ideal oracle (simulated)")
    marked = int((costs == 0).sum())
    k = grover_iterations(len(costs), marked)
    probs = grover_probs(costs, k)
    st["samplers"].append(summarise(f"Grover k={k}", cubes, costs, probs, trials, rng))
    calls = np.array([bbht_oracle_calls(len(costs), marked, rng) for _ in range(2000)])
    st["grover_k"], st["bbht"] = k, calls.mean()
    print(f"      unknown number of solutions (BBHT): {calls.mean():.1f} oracle calls on average "
          f"to get one valid 2-factor (known-count Grover uses {k} per attempt)")

    print("\nClassical stage on Qiskit-style counts (decoder round trip)")
    f, g = needed_pair(cubes)
    assert decode_key(encode_key(f), n) == f, "decoder failed on a known answer"
    draws = rng.choice(len(probs), size=1000, p=probs / probs.sum())
    counts = {}
    for d in draws:
        key = encode_key(tuple(int(x) for x in np.unravel_index(d, (3,) * n)))
        counts[key] = counts.get(key, 0) + 1
    towers, seen = assemble_from_counts(cubes, counts)
    print(f"  1000 Grover shots -> {seen} distinct 2-factors seen, "
          f"{len(towers)} verified tower(s): {towers[0] if towers else None}")
    st["seen"], st["verified"] = seen, len(towers)

    st["p_random"] = (costs == 0).mean()                      # random guessing baseline
    uniform = np.full(len(costs), 1 / len(costs))
    runs = shots_to_tower(cubes, costs, uniform, trials, rng)
    ok = np.isfinite(runs)
    st["random_runs"] = {"solved": ok.mean(), "median": np.median(runs) if ok.any() else np.inf,
                         "p95": np.sort(runs)[int(0.95 * len(runs)) - 1],
                         "within": {s: (runs <= s).mean() for s in (10, 50, 100)}}
    print("\nReminder: the circuit's job here is sampling 2-factors; brute force and the graph "
          "method are instant at this size.")
    return st


def run_scale(rng):
    print("\n" + "=" * 78)
    print("Scaling with N (unique-answer seeds): shots to see both needed 2-factors, 95%")
    print("=" * 78)
    print(f"{'N':>2} {'qubits':>7} {'K':>3} {'3^N':>6} | {'QAOA p=3':>22} | {'Grover':>26}")
    for n, seed in BONUS.items():
        cubes = make_puzzle(n, seed)
        pair = needed_pair(cubes)
        if pair is None:
            print(f"{n:>2}  (seed {seed} does not have exactly one disjoint pair; skipped)")
            continue
        costs = fb_costs(cubes)
        idx = [int(np.ravel_multi_index(f, (3,) * n)) for f in pair]
        factors = two_factors(cubes)
        angles = tune_qaoa(costs, n, pmax=3, rng=rng)
        pq = qaoa_runner(costs, n)(*angles[3])
        k = grover_iterations(len(costs), len(factors))
        pg = grover_probs(costs, k)
        sq = shots_for_confidence(pq[idx[0]], pq[idx[1]])
        sg = shots_for_confidence(pg[idx[0]], pg[idx[1]])
        q_txt = (f"{pq[idx].sum():.1%} -> {sq} shots" if np.isfinite(sq)
                 else f"{pq[idx].sum():.1%} -> fails")
        g_txt = f"k={k}, {pg[idx].sum():.0%} -> {sg} shots"
        print(f"{n:>2} {3 * n:>7} {len(factors):>3} {3 ** n:>6,} | {q_txt:>22} | {g_txt:>26}")
    print("\nQAOA column: P is the chance one shot lands on either needed factor. A 'fails' entry\n"
          "means the tuned circuit put essentially no weight on them; that is a tuning result for\n"
          "this optimiser budget (p<=3, few restarts), not a proven limit of the method.")


def main():
    parser = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    parser.add_argument("--scale", action="store_true", default=DEFAULT_SCALE, help="also run N=5..8")
    parser.add_argument("--trials", type=int, default=1000, help="simulated runs per sampler")
    parser.add_argument("--seed", type=int, default=SEED)
    parser.add_argument("--cubes", type=int, default=DEFAULT_CUBES, choices=[4, 5, 6, 7, 8],
                        help="number of cubes for the main run (8 = double the starter puzzle)")
    args, _ = parser.parse_known_args()   # ignores extra args a notebook/IDE may pass
    rng = np.random.default_rng(args.seed)
    print(f"seed {args.seed}\n")
    stats = run_n4(args.trials, rng, args.cubes)
    if args.scale:
        run_scale(rng)
    print_slide_stats(stats)


if __name__ == "__main__":
    main()